## memo

- URL: https://www.kaggle.com/code/koyamaryuji/qwen-array-task-inference/notebook?scriptVersionId=344108585
- v3_ood
- base_model

In [ ]:
import polars as pl
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent.parent))
from src.gen_task import RAW_RULES, RAW_OOD_RULES_V1

# RULE_CONFIG = RAW_RULES
RULE_CONFIG = RAW_OOD_RULES_V1

RULES = [
    {
        "id": f"{i:03d}",
        "primitives": rule,
    }
    for i, rule in enumerate(RULE_CONFIG)
]

df = pl.read_csv(Path("debug_predictions.csv"))
df = df.sort("id")
stop = 0
for pred in df.iter_rows(named=True):
    print("###" * 50)
    print(f'Task ID: \n{pred["id"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Prompt: \n{pred["raw_prompt"].encode().decode("unicode-escape")}')
    print("===" * 50)
    print(f'Raw_output: \n{pred["raw_output"]}')
    print("===" * 50)
    print(f'Target: \n{pred["target"]}')
    print("===" * 50)
    print(f'Finish: \n{pred["finish_reason"]}')
    print("===" * 50)
    print(f'tokens: \n{pred["num_tokens"]}')
    if pred["finish_reason"] == "stop":
        stop += 1

print(stop)


######################################################################################################################################################
Task ID: 
000_0
Prompt: 

Infer the transformation rule from examples.
Output the final array.
        
Example 1

Input:
[1, 0, 4, 3, 3, 2, 1, 8, 1, 9]

Output:
[0, 9, 1, 0, 4, 3, 3, 2, 1, 8, 1]

Example 2

Input:
[0, 0, 1, 3, 3, 8, 9, 0]

Output:
[0, 0, 0, 0, 1, 3, 3, 8, 9]

Example 3

Input:
[3, 8, 6, 3, 7, 9, 4, 0, 2]

Output:
[0, 2, 3, 8, 6, 3, 7, 9, 4, 0]

Example 4

Input:
[6, 5, 4, 2, 3, 5, 1, 1, 6, 1]

Output:
[0, 1, 6, 5, 4, 2, 3, 5, 1, 1, 6]

Example 5

Input:
[5, 9, 4, 0, 7, 8, 1]

Output:
[0, 1, 5, 9, 4, 0, 7, 8]

Query

Input:
[1, 8, 4, 9, 5, 9, 3, 1]

Output:
Raw_output: 
<think>
Okay, let's try to figure out the transformation rule from these examples. So, the task is to take an input array and apply some transformation to get the output array. Let me look at the examples one by one and see if I can spot a pattern.

Start

In [2]:
import re
import ast
import polars as pl
from pathlib import Path


def extract_answer(text):
    if text is None:
        return 'NOT_FOUND'

    matches = re.findall(r'\[[^\[\]]*\]', text)
    arrays = []
    for match in matches:
        try:
            value = ast.literal_eval(match)

            if isinstance(value, list):
                arrays.append(value)

        except (ValueError, SyntaxError):
            pass

    if arrays == []:
        return 'NOT_FOUND'

    return str(arrays[-1])

df = pl.read_csv(Path("debug_predictions.csv"))

match_count = 0
match = []
miss = []
for pred in df.iter_rows(named=True):
    # print("###" * 50)
    answer = extract_answer(pred["raw_output"])
    if answer == pred["target"]:
        # print(pred["id"])
        match_count += 1
        match.append(pred["id"].split("_")[0])
    else:
        miss.append(pred["id"].split("_")[0])
        # print(pred["id"])
        # print(pred["raw_prompt"].encode().decode("unicode-escape"))
        # print(pred["target"])
print(f"len(df): {len(df)}")
print(f"match_count: {match_count}")
print(f"acc: {match_count / len(df)}")
# print(miss)
from collections import Counter


counts = Counter(miss)
match_counts = Counter(match)

len(df): 480
match_count: 67
acc: 0.13958333333333334


# 正解

In [ ]:
match_results = []

for rule in RULES:
    if rule["id"] in [i for i, j in match_counts.items()]:
        for i, j in match_counts.items():
            if rule["id"] == i:
                match_results.append({"task_id": i, "count": j, "rule": rule["primitives"]})
    else:
        match_results.append({"task_id": rule["id"], "count": 0, "rule": rule["primitives"]})

match_results = sorted(match_results, key=lambda x: x["task_id"], reverse=True)

for x in match_results:
    print(x)

records = []
for x in match_results:
    records.append({"task_id": x["task_id"], "rule": x["rule"], "count": x["count"]})
    print(x)

heatmap_df = pl.DataFrame(records)
pl.Config.set_tbl_rows(-1)

heatmap_df

{'task_id': '047', 'count': 0, 'rule': ['swap_first_last', 'take_odd_positions']}
{'task_id': '046', 'count': 0, 'rule': ['swap_first_last', 'take_even_positions']}
{'task_id': '045', 'count': 0, 'rule': ['swap_first_last', 'mirror']}
{'task_id': '044', 'count': 0, 'rule': ['swap_first_last', 'adjacent_sum']}
{'task_id': '043', 'count': 0, 'rule': ['swap_first_last', 'subtract_next']}
{'task_id': '042', 'count': 0, 'rule': ['swap_first_last', 'mod_3']}
{'task_id': '041', 'count': 0, 'rule': ['swap_first_last', 'mod_2']}
{'task_id': '040', 'count': 0, 'rule': ['swap_first_last', 'add_3']}
{'task_id': '039', 'count': 0, 'rule': ['swap_first_last', 'add_2']}
{'task_id': '038', 'count': 0, 'rule': ['swap_first_last', 'add_1']}
{'task_id': '037', 'count': 0, 'rule': ['swap_first_last', 'multiply_3']}
{'task_id': '036', 'count': 0, 'rule': ['swap_first_last', 'multiply_2']}
{'task_id': '035', 'count': 2, 'rule': ['swap_first_last', 'pop_left']}
{'task_id': '034', 'count': 8, 'rule': ['swap_f

# 不正解

In [4]:
results = []
for i, j in counts.items():
    for rule in RULES:
        if rule["id"] == i:
            # print(i, j, rule)
            results.append({"task_id": i, "count": j, "rule": rule["primitives"]})

results = sorted(results, key=lambda x: x["task_id"], reverse=True)
results

[{'task_id': '047',
  'count': 10,
  'rule': ['swap_first_last', 'take_odd_positions']},
 {'task_id': '046',
  'count': 10,
  'rule': ['swap_first_last', 'take_even_positions']},
 {'task_id': '045', 'count': 10, 'rule': ['swap_first_last', 'mirror']},
 {'task_id': '044', 'count': 10, 'rule': ['swap_first_last', 'adjacent_sum']},
 {'task_id': '043', 'count': 10, 'rule': ['swap_first_last', 'subtract_next']},
 {'task_id': '042', 'count': 10, 'rule': ['swap_first_last', 'mod_3']},
 {'task_id': '041', 'count': 10, 'rule': ['swap_first_last', 'mod_2']},
 {'task_id': '040', 'count': 10, 'rule': ['swap_first_last', 'add_3']},
 {'task_id': '039', 'count': 10, 'rule': ['swap_first_last', 'add_2']},
 {'task_id': '038', 'count': 10, 'rule': ['swap_first_last', 'add_1']},
 {'task_id': '037', 'count': 10, 'rule': ['swap_first_last', 'multiply_3']},
 {'task_id': '036', 'count': 10, 'rule': ['swap_first_last', 'multiply_2']},
 {'task_id': '035', 'count': 8, 'rule': ['swap_first_last', 'pop_left']},
 